# QLoRA Fine-Tuning on SageMaker - AWS Hi-Tech, Electronics, & Semiconductor Compute

Fine-tunes `meta-llama/Llama-3.1-8B-Instruct` on facts about AWS's compute stack for its Hi-Tech, Electronics, and Semiconductor vertical (NVIDIA/AMD-based EC2 instances, AWS Trainium/Inferentia, EDA-relevant infrastructure), using QLoRA on Amazon SageMaker.

Pipeline: augment seed Q&A via Bedrock Claude Haiku &rarr; upload to S3 &rarr; launch a SageMaker training job (`scripts/train.py`) &rarr; deploy the fine-tuned model to a SageMaker endpoint &rarr; evaluate against the base model using Bedrock Claude Opus as an LLM judge &rarr; clean up.

**Before running:** request access to `meta-llama/Llama-3.1-8B-Instruct` on Hugging Face, create a Read token, and enable Bedrock model access for Claude Haiku and Claude Opus in your AWS account.

**Data quantity note:** `data/seed_qa.jsonl` ships with a small starter set. Factual recall quality depends much more on data volume than on model size or hyperparameters. Aim for **100-150 hand-written seed examples** before augmentation, not the ~20 starters included here. If evaluation scores come back low, add more seed facts before tuning anything else.

## 1. Hugging Face login and install requirements

Token is entered interactively and never written to disk or committed.

In [1]:
from getpass import getpass
from huggingface_hub import login

hf_token = getpass("Hugging Face token: ")
login(token=hf_token)

In [2]:
!pip install -q -r requirements.txt --upgrade

## 2. Augment seed data via Bedrock Claude Haiku

The augmentation prompt is deliberately strict. Unconstrained augmentation lets the model "helpfully" invent details like a different GPU memory figure, a renamed instance type, or an extra chip vendor. Every invented fact gets learned by the fine-tuned model as ground truth. The prompt below forbids adding any fact not present in the original and requires exact preservation of names and numbers.

In [3]:
import json
import re

import boto3

bedrock = boto3.client("bedrock-runtime")
seeds = [json.loads(line) for line in open("data/seed_qa.jsonl")]

AUGMENT_PROMPT = """You are a training-data generator for a fine-tuned LLM about AWS compute
for the hi-tech, electronics, and semiconductor industry: AWS's NVIDIA- and AMD-based EC2
instances, AWS's own Trainium/Inferentia/Graviton silicon, and AWS infrastructure relevant to
chip design and EDA workloads.

Given one Q&A pair, produce {n} PARAPHRASED variants.

STRICT RULES:
1. Every answer must contain ONLY facts from the original answer.
2. Do not add, invent, or embellish any detail not present in the original.
3. Preserve every proper noun EXACTLY as written: instance names (e.g. P5, G5, Trn1),
   chip names (e.g. H100, A100, A10G, Trainium, Inferentia, Graviton, EPYC), company names
   (NVIDIA, AMD, Xilinx, Annapurna Labs), and SDK/tool names (Neuron SDK, ParallelCluster, EFA).
4. Preserve every number EXACTLY as written (GPU counts, memory sizes, generation numbers).
5. Rephrase the QUESTION in a different way each time (different wording, same meaning).
6. Output ONLY a JSON array, no commentary, no markdown code fences.

Original question: {q}
Original answer: {a}

Return a JSON array of {n} objects, each with "question" and "answer" string fields."""

SYSTEM_PROMPT = "You are an AWS compute expert for the hi-tech, electronics, and semiconductor industry."


def extract_json_array(text):
    match = re.search(r"\[.*\]", text, re.DOTALL)
    return json.loads(match.group(0) if match else text)


def augment(seed, n=5):
    seed_q = seed["messages"][1]["content"]
    seed_a = seed["messages"][2]["content"]
    prompt = AUGMENT_PROMPT.format(n=n, q=seed_q, a=seed_a)
    resp = bedrock.invoke_model(
        modelId="us.anthropic.claude-haiku-4-5-20251001-v1:0",
        body=json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 2000,
            "temperature": 0.3,
            "messages": [{"role": "user", "content": prompt}],
        }),
    )
    text = json.loads(resp["body"].read())["content"][0]["text"]
    return extract_json_array(text)


augmented = list(seeds)
for seed in seeds:
    for variant in augment(seed):
        augmented.append({
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": variant["question"]},
                {"role": "assistant", "content": variant["answer"]},
            ]
        })

print(f"{len(seeds)} seeds -> {len(augmented)} augmented examples")

## 3. Split and upload to S3

In [4]:
import sagemaker
from datasets import Dataset

session = sagemaker.Session()
bucket = "rp-sagemaker-hf-ft-bucket"
role = sagemaker.get_execution_role()

eval_set = [json.loads(line) for line in open("data/eval.jsonl")]

split = Dataset.from_list(augmented).train_test_split(test_size=0.1, seed=42)
split["train"].to_json("train.jsonl")
split["test"].to_json("val.jsonl")

train_s3 = session.upload_data("train.jsonl", bucket=bucket, key_prefix="qlora-semiconductor/data")
val_s3 = session.upload_data("val.jsonl", bucket=bucket, key_prefix="qlora-semiconductor/data")
print(train_s3)
print(val_s3)

## 4. Launch the SageMaker training job

Runs `scripts/train.py` inside a Hugging Face Deep Learning Container on a separate `ml.g5.2xlarge` instance (A10G, 24 GB). QLoRA loads the 8B model in 4-bit Normal Float 4-bit (NF4), trains LoRA adapters, then performs the two-step merge (adapter saved in fp16, base model reloaded in fp16, then merged) so the artifact saved to S3 is a standalone model with no PEFT dependency at inference time.

In [6]:
from sagemaker.huggingface import HuggingFace

estimator = HuggingFace(
    entry_point="train.py",
    source_dir="scripts/",
    role=role,
    transformers_version="4.49.0",
    pytorch_version="2.5.1",
    py_version="py311",
    instance_type="ml.g5.2xlarge",
    instance_count=1,
    environment={"HF_TOKEN": hf_token},
    hyperparameters={
        "model_id": "meta-llama/Llama-3.1-8B-Instruct",
        "epochs": 5,
        "lora_r": 16,
        "lora_alpha": 32,
    },
)

estimator.fit(inputs={"train": train_s3, "val": val_s3}, wait=True)

## 5. Deploy the fine-tuned model

Deploys the fine-tuned model to a SageMaker endpoint. The base model is evaluated via Bedrock in section 6 (no second endpoint needed). Remember to delete this endpoint when done to avoid ongoing charges.

In [27]:
import sagemaker
import boto3
from sagemaker.huggingface import HuggingFaceModel, get_huggingface_llm_image_uri

# Text-Generation Inference (TGI) container — handles timeouts correctly unlike the standard HF DLC
tgi_image_uri = get_huggingface_llm_image_uri("huggingface", version="2.4.0")

ft_model = HuggingFaceModel(
    model_data=estimator.model_data,
    role=role,
    image_uri=tgi_image_uri,
    env={
        "HF_MODEL_ID": "/opt/ml/model",  # load from S3 artifact, not HuggingFace Hub
        "HF_TASK": "text-generation",
        "MAX_INPUT_LENGTH": "2048",
        "MAX_TOTAL_TOKENS": "4096",
        "MAX_BATCH_PREFILL_TOKENS": "2048",
    },
)

# ml.g5.2xlarge skipped — confirmed InsufficientInstanceCapacity in us-east-1.
# Priority-ordered fallbacks, all with sufficient VRAM for Llama-3.1-8B with TGI:
#   ml.g5.4xlarge   — A10G 24 GB, more CPU/RAM than 2xlarge
#   ml.g5.12xlarge  — 4x A10G 96 GB
#   ml.g4dn.2xlarge — T4 16 GB, more CPU/RAM than xlarge (tight but workable for 8B fp16)
#   ml.g4dn.xlarge  — T4 16 GB (slower but widely available)
INSTANCE_PRIORITY = [
    "ml.g5.4xlarge",
    "ml.g5.12xlarge",
    "ml.g4dn.2xlarge",
    "ml.g4dn.xlarge",
]

sm_client = boto3.client("sagemaker")

ft_predictor = None
for instance_type in INSTANCE_PRIORITY:
    # Clean up any leftover endpoint / config from a previous failed attempt
    for resource, delete_fn in [
        ("endpoint", sm_client.delete_endpoint),
        ("endpoint-config", sm_client.delete_endpoint_config),
    ]:
        try:
            if resource == "endpoint":
                delete_fn(EndpointName="qlora-semiconductor-finetuned")
            else:
                delete_fn(EndpointConfigName="qlora-semiconductor-finetuned")
            print(f"Cleaned up existing {resource}.")
        except sm_client.exceptions.ClientError:
            pass  # Nothing to clean up

    print(f"Trying {instance_type} ...")
    try:
        ft_predictor = ft_model.deploy(
            initial_instance_count=1,
            instance_type=instance_type,
            endpoint_name="qlora-semiconductor-finetuned",
            container_startup_health_check_timeout=900,
        )
        print(f"Fine-tuned endpoint deployed on {instance_type} with TGI container.")
        break
    except Exception as e:
        if "InsufficientInstanceCapacity" in str(e):
            print(f"  No capacity for {instance_type}, trying next ...")
        else:
            raise  # Unexpected error — surface it immediately

if ft_predictor is None:
    raise RuntimeError("All instance types exhausted. Try again later or choose a different region.")

In [ ]:
import boto3
sm = boto3.client("sagemaker")
resp = sm.describe_endpoint(EndpointName="qlora-semiconductor-finetuned")
print(f"Status: {resp['EndpointStatus']}")
if resp.get("FailureReason"):
    print(f"Failure reason: {resp['FailureReason']}")

## 6. Set up base model for comparison

Uses Bedrock to call the un-fine-tuned `meta-llama/Llama-3.1-8B-Instruct` directly — no second endpoint needed. The tokenizer is loaded locally for prompt formatting only.

In [14]:
from transformers import AutoTokenizer
import boto3
from botocore.config import Config

base_model_id = "meta-llama/Llama-3.1-8B-Instruct"

# Tokenizer used only for prompt formatting
base_tokenizer = AutoTokenizer.from_pretrained(base_model_id, token=hf_token)

# Set 10-minute read timeout on the boto3 client to match server-side timeout
sm_runtime = boto3.client(
    "sagemaker-runtime",
    config=Config(read_timeout=600, connect_timeout=10),
)


def build_messages(question):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]


def base_generate(question, max_new_tokens=200):
    """Call base Llama 3.1 8B via Bedrock — no endpoint needed."""
    prompt = base_tokenizer.apply_chat_template(
        build_messages(question), add_generation_prompt=True, tokenize=False,
    )
    response = bedrock.invoke_model(
        modelId="us.meta.llama3-1-8b-instruct-v1:0",
        body=json.dumps({"prompt": prompt, "max_gen_len": max_new_tokens}),
    )
    return json.loads(response["body"].read())["generation"].strip()


def ft_generate(question, max_new_tokens=200):
    """Call fine-tuned endpoint via boto3 with 10-minute read timeout."""
    prompt = base_tokenizer.apply_chat_template(
        build_messages(question), add_generation_prompt=True, tokenize=False,
    )
    response = sm_runtime.invoke_endpoint(
        EndpointName="qlora-semiconductor-finetuned",
        ContentType="application/json",
        Body=json.dumps({"inputs": prompt, "parameters": {"max_new_tokens": max_new_tokens}}),
    )
    result = json.loads(response["Body"].read())
    return result[0]["generated_text"][len(prompt):].strip()


print("base_generate (Bedrock) and ft_generate (endpoint) ready.")

In [15]:
# Spot-check both models before running full evaluation
test_question = "What GPU does the ml.g5.2xlarge instance use?"

print("=== BASE MODEL ANSWER (Bedrock) ===")
print(base_generate(test_question))

print("\n=== FINE-TUNED MODEL ANSWER (Endpoint) ===")
print(ft_generate(test_question))

## 7. Evaluate with Bedrock Claude Opus as LLM judge

Factual-recall answers are generative and get paraphrased even when correct, so n-gram metrics like ROUGE/BLEU penalize correct answers, and exact-match will read as a near-constant 0.0 regardless of quality. An LLM judge (Claude Opus, `temperature=0` for reproducible scores) evaluates whether the required facts are present, the way a domain expert would.

In [ ]:
JUDGE_PROMPT = """Score the RESPONSE 1-5 against the REFERENCE.
5 - Fully correct, all key facts present
4 - Mostly correct, minor omissions
3 - Partially correct, misses key details
2 - Mostly wrong, one relevant element
1 - Completely wrong or hallucinated

Return ONLY JSON: {{"score": <1-5>, "reason": "<one sentence>"}}

QUESTION: {q}
REFERENCE: {ref}
RESPONSE: {resp}"""


def judge(question, reference, response):
    body = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 200,
        "temperature": 0,
        "messages": [{"role": "user", "content": JUDGE_PROMPT.format(q=question, ref=reference, resp=response)}],
    })
    out = bedrock.invoke_model(modelId="us.anthropic.claude-opus-4-1-20250805-v1:0", body=body)
    text = json.loads(out["body"].read())["content"][0]["text"]
    return extract_json_array(f"[{text}]")[0]


results = []
for ex in eval_set:
    question = ex["messages"][1]["content"]
    reference = ex["messages"][2]["content"]

    base_answer = base_generate(question)
    ft_answer = ft_generate(question)

    base_judged = judge(question, reference, base_answer)
    ft_judged = judge(question, reference, ft_answer)

    results.append({
        "question": question,
        "base_answer": base_answer,
        "base_score": base_judged["score"],
        "ft_answer": ft_answer,
        "ft_score": ft_judged["score"],
    })

import statistics

base_avg = statistics.mean(r["base_score"] for r in results)
ft_avg = statistics.mean(r["ft_score"] for r in results)
print(f"base model avg score:       {base_avg:.2f} / 5")
print(f"fine-tuned model avg score: {ft_avg:.2f} / 5")

## 8. Cleanup

The fine-tuned endpoint bills hourly while running. Run this cell as soon as you're done evaluating to avoid unnecessary charges. The base model uses Bedrock pay-per-token — no endpoint to delete.

In [ ]:
import boto3
boto3.client("sagemaker").delete_endpoint(EndpointName="qlora-semiconductor-finetuned")
print("Fine-tuned endpoint deleted.")
# Note: base model uses Bedrock — no endpoint to delete.